Load the Master Dataset

In [0]:
df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .load("/Volumes/workspace/default/superstore_data/Sample - Superstore.csv")

# display(df)

In [0]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [0]:
print("Number of rows:", df.count())
print("Number of columns:", len(df.columns))

Number of rows: 9994
Number of columns: 21


 Data Cleaning

In [0]:
from pyspark.sql.functions import col, when, count

null_df = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

display(null_df)

Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
clean_df = df.dropDuplicates()

print("Original Rows :", df.count())
print("Rows after removing duplicates :", clean_df.count())

Original Rows : 9994
Rows after removing duplicates : 9994


We need to replace all spaces or other character with underscore as well as remove parenthesis as the newer versions of Delta Lake reject these by default

In [0]:
new_columns = [
    c.replace(" ", "_")
     .replace("-", "_")
     .replace("/", "_")
     .replace("(", "")
     .replace(")", "")
    for c in clean_df.columns
]

clean_df = clean_df.toDF(*new_columns)

clean_df.printSchema()

root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



Store the Dataset as a Delta Table

In [0]:
clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/superstore_data/delta_superstore")

In [0]:
delta_df = spark.read.format("delta").load(
    "/Volumes/workspace/default/superstore_data/delta_superstore"
)

verifying that the delta table doesn't have any duplicates

In [0]:
delta_df.groupBy("Row_ID") \
    .count() \
    .filter("count > 1") \
    .show()

+------+-----+
|Row_ID|count|
+------+-----+
+------+-----+



In [0]:
delta_df.printSchema()

root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



Loading the Incremental Dataset

In [0]:
incremental_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .load("/Volumes/workspace/default/superstore_data/sample_superstore_incremental.csv")

Validate Schema Compatibility

In [0]:
incremental_df.printSchema()

root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



MERGE Operation

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(
    spark,
    "/Volumes/workspace/default/superstore_data/delta_superstore"
)

(delta_table.alias("target")
 .merge(
     incremental_df.alias("source"),
     "target.Row_ID = source.Row_ID"
 )
 .whenMatchedUpdateAll()
 .whenNotMatchedInsertAll()
 .execute())

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

Validate the Results

In [0]:
final_df = spark.read.format("delta").load(
    "/Volumes/workspace/default/superstore_data/delta_superstore"
)

print("Final row count:", final_df.count())

Final row count: 9996


In [0]:
# display(final_df)

Validating for Duplicate records

In [0]:
final_df.groupBy("Row_ID") \
    .count() \
    .filter("count > 1") \
    .show()

+------+-----+
|Row_ID|count|
+------+-----+
+------+-----+



Summary
- Loaded the Sample Superstore dataset into Spark.
- Performed data cleaning by checking null values, removing duplicates, and renaming columns.
- Stored the cleaned data as a Delta table.
- Created an incremental dataset with 3 updates and 2 new records.
- Applied the MERGE operation to update existing records and insert new ones.
- Validated the final output by checking the row count and confirming successful updates and inserts.
-  Demonstrated how Delta Lake simplifies incremental data processing using a single MERGE operation while maintaining data consistency.